# Práctica · Verificación de firmas manuscritas offline
**Procesamiento de Imágenes, Audio y Vídeo** · Grado en Ciencia e Ingeniería de Datos

Autores: _Nombre Apellido_ y _Nombre Apellido_

Índice:
0. Datos y protocolo
1. Funciones de evaluación biométrica (FAR, FRR, EER, DET)
2. Sistema paramétrico con OpenCV
3. Sistema de aprendizaje automático (SigNet simplificado)
4. Comparación en evaluación
5. Prueba con usuarios nuevos

## 0. Datos y protocolo
Base de datos CEDAR (tal y como se proporciona en el Campus Virtual): 24 genuinas y 24 falsificaciones hábiles por usuario.

| Conjunto | Uso |
|---|---|
| Genuinas 1–10 | Referencias → patrón del usuario |
| Genuinas 11–24 | Pruebas genuinas → FRR |
| Falsificaciones 1–24 | Pruebas impostoras → FAR |

Usuarios de **desarrollo**: 1–30 (se ajustan hiperparámetros y umbral). Usuarios de **evaluación**: el resto (solo se mide).

In [ ]:
import os, re, glob
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from scipy.stats import norm

DATA_DIR = "data"
DIR_GEN = os.path.join(DATA_DIR, "genuinas", "full_org", "full_org")
DIR_FORG = os.path.join(DATA_DIR, "falsificadas", "full_forg", "full_forg")

N_REF = 10  # genuinas 1..10 = referencias

def listar_firmas(dir_gen, dir_forg):
    """Devuelve un DataFrame con usuario, n.º de firma, tipo y ruta de cada imagen."""
    filas = []
    for d, tipo in [(dir_gen, "genuina"), (dir_forg, "falsificada")]:
        for f in glob.glob(os.path.join(d, "*.png")):
            m = re.match(r"[a-z]+_(\d+)_(\d+)\.png$", os.path.basename(f))
            if m:
                filas.append(dict(usuario=int(m.group(1)), firma=int(m.group(2)), tipo=tipo, ruta=f))
    df = pd.DataFrame(filas).sort_values(["usuario", "tipo", "firma"]).reset_index(drop=True)
    # rol de cada firma dentro del protocolo
    df["rol"] = np.where((df.tipo == "genuina") & (df.firma <= N_REF), "referencia", "prueba")
    return df

firmas = listar_firmas(DIR_GEN, DIR_FORG)
usuarios = sorted(firmas.usuario.unique())
USR_DEV = [u for u in usuarios if u <= 30]
USR_EVAL = [u for u in usuarios if u > 30]
print(f"{len(usuarios)} usuarios | desarrollo: {len(USR_DEV)} | evaluación: {len(USR_EVAL)}")
print(firmas.groupby(["tipo", "rol"]).size())

In [ ]:
def leer_gris(ruta):
    """Algunas imágenes de CEDAR vienen en color y otras en gris: se leen todas en gris."""
    return cv2.imread(ruta, cv2.IMREAD_GRAYSCALE)

fig, axs = plt.subplots(2, 4, figsize=(14, 5))
for i, ax in enumerate(axs[0]):
    ax.imshow(leer_gris(firmas.query("usuario==1 and tipo=='genuina'").ruta.iloc[i]), cmap="gray"); ax.set_title(f"Genuina {i+1}"); ax.axis("off")
for i, ax in enumerate(axs[1]):
    ax.imshow(leer_gris(firmas.query("usuario==1 and tipo=='falsificada'").ruta.iloc[i]), cmap="gray"); ax.set_title(f"Falsificación {i+1}"); ax.axis("off")
plt.suptitle("Usuario 1"); plt.tight_layout(); plt.show()

## 1. Funciones de evaluación biométrica
Ambos sistemas producen una **puntuación de similitud** (mayor = más parecida al patrón). Para un umbral $t$:
- **FAR(t)** = % de falsificaciones con puntuación $\ge t$ (impostores aceptados).
- **FRR(t)** = % de genuinas de prueba con puntuación $< t$ (genuinos rechazados).
- **EER**: punto en que FAR = FRR.
- **Curva DET**: FRR frente a FAR en escala normal (probit).

In [ ]:
def far_frr(scores_gen, scores_imp, umbral):
    far = np.mean(np.asarray(scores_imp) >= umbral)
    frr = np.mean(np.asarray(scores_gen) < umbral)
    return far, frr

def curva_far_frr(scores_gen, scores_imp):
    """FAR y FRR para todos los umbrales posibles (las propias puntuaciones)."""
    sg, si = np.asarray(scores_gen), np.asarray(scores_imp)
    umbrales = np.unique(np.concatenate([sg, si, [np.inf]]))
    far = np.array([np.mean(si >= t) for t in umbrales])
    frr = np.array([np.mean(sg < t) for t in umbrales])
    return umbrales, far, frr

def eer(scores_gen, scores_imp):
    """Devuelve (EER, umbral del EER)."""
    umbrales, far, frr = curva_far_frr(scores_gen, scores_imp)
    i = np.argmin(np.abs(far - frr))
    return (far[i] + frr[i]) / 2, umbrales[i]

def separar(df_scores):
    g = df_scores.loc[df_scores.tipo == "genuina", "puntuacion"].values
    f = df_scores.loc[df_scores.tipo == "falsificada", "puntuacion"].values
    return g, f

def resumen(df_scores, umbral=None, nombre=""):
    g, f = separar(df_scores)
    e, t_eer = eer(g, f)
    txt = f"{nombre}: EER = {100*e:.2f} % (umbral EER = {t_eer:.4f})"
    if umbral is not None:
        far, frr = far_frr(g, f, umbral)
        txt += f" | con umbral fijo {umbral:.4f}: FAR = {100*far:.2f} %, FRR = {100*frr:.2f} %"
    print(txt)
    return e, t_eer

def plot_det(sistemas, titulo="Curvas DET", ax=None):
    """sistemas: dict nombre -> DataFrame de puntuaciones."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    eps = 1e-3
    for nombre, df in sistemas.items():
        g, f = separar(df)
        _, far, frr = curva_far_frr(g, f)
        e, _ = eer(g, f)
        ax.plot(norm.ppf(np.clip(far, eps, 1 - eps)), norm.ppf(np.clip(frr, eps, 1 - eps)),
                lw=2, label=f"{nombre} (EER = {100*e:.1f} %)")
    ticks = np.array([0.1, 0.5, 1, 2, 5, 10, 20, 40, 60, 80, 95]) / 100
    ax.set_xticks(norm.ppf(ticks)); ax.set_xticklabels([f"{100*t:g}" for t in ticks])
    ax.set_yticks(norm.ppf(ticks)); ax.set_yticklabels([f"{100*t:g}" for t in ticks])
    lim = norm.ppf([eps, 0.95]); ax.set_xlim(lim); ax.set_ylim(lim)
    ax.plot(lim, lim, "k--", lw=0.8, label="FAR = FRR")
    ax.set_xlabel("FAR (%)"); ax.set_ylabel("FRR (%)"); ax.set_title(titulo)
    ax.grid(alpha=0.3); ax.legend()
    return ax

## 2. Sistema paramétrico con OpenCV
### 2.1 Preprocesado
1. Lectura en gris.
2. **Filtro de mediana** 3×3 para eliminar el ruido de escaneo (puntitos aislados).
3. **Binarización de Otsu** invertida (tinta = 1, fondo = 0).
4. **Eliminación de componentes conexas diminutas** (< 10 píxeles).
5. **Recorte** a la caja que contiene la firma.

In [ ]:
MIN_AREA_CC = 10  # componentes conexas con menos píxeles se consideran ruido

def preprocesar(img):
    img = cv2.medianBlur(img, 3)
    _, b = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    # eliminar manchas pequeñas
    n, lab, stats, _ = cv2.connectedComponentsWithStats(b, connectivity=8)
    limpia = np.zeros_like(b)
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] >= MIN_AREA_CC:
            limpia[lab == i] = 255
    ys, xs = np.nonzero(limpia)
    if len(xs) == 0:
        return limpia
    return limpia[ys.min():ys.max() + 1, xs.min():xs.max() + 1]

ej = firmas.query("usuario==1 and tipo=='genuina'").ruta.iloc[0]
fig, axs = plt.subplots(1, 2, figsize=(10, 3))
axs[0].imshow(leer_gris(ej), cmap="gray"); axs[0].set_title("Original")
axs[1].imshow(preprocesar(leer_gris(ej)), cmap="gray"); axs[1].set_title("Otsu + limpieza + recorte")
for a in axs: a.axis("off")
plt.show()

### 2.2 Características
Sobre la imagen binaria recortada se extraen características geométricas sencillas e independientes de la escala:

| # | Característica | Descripción |
|---|---|---|
| 1 | Densidad de tinta | píxeles de tinta / área de la caja |
| 2 | Relación ancho/alto | proporción de la caja (en log) |
| 3–4 | Centroide (x, y) | centro de masas normalizado por ancho y alto |
| 5 | Componentes conexas | n.º de trazos separados |
| 6–8 | Momentos de Hu 1–3 | forma global, en escala logarítmica |
| 9–12 | Tinta por cuadrantes | fracción de tinta en cada cuadrante respecto al centroide |
| 13 | Inclinación | ángulo del eje principal de inercia |

In [ ]:
NOMBRES_CAR = ["densidad", "log_aspecto", "cx", "cy", "n_cc", "hu1", "hu2", "hu3",
               "q_si", "q_sd", "q_ii", "q_id", "inclinacion"]

def caracteristicas(b):
    h, w = b.shape
    t = (b > 0).astype(np.uint8)
    total = t.sum()
    densidad = total / (h * w)
    log_aspecto = np.log(w / h)
    m = cv2.moments(t, binaryImage=True)
    cx, cy = m["m10"] / m["m00"], m["m01"] / m["m00"]
    n_cc = cv2.connectedComponents(t, connectivity=8)[0] - 1
    hu = cv2.HuMoments(m).flatten()[:3]
    hu = -np.sign(hu) * np.log10(np.abs(hu) + 1e-30)
    icx, icy = int(round(cx)), int(round(cy))
    q = np.array([t[:icy, :icx].sum(), t[:icy, icx:].sum(), t[icy:, :icx].sum(), t[icy:, icx:].sum()]) / total
    inclinacion = 0.5 * np.arctan2(2 * m["mu11"], m["mu20"] - m["mu02"])
    return np.array([densidad, log_aspecto, cx / w, cy / h, n_cc, *hu, *q, inclinacion], dtype=float)

# extracción para todas las firmas (se guarda en caché para no repetirla)
CACHE_PAR = "cache_caracteristicas_param.npy"
if os.path.exists(CACHE_PAR):
    X_par = np.load(CACHE_PAR)
else:
    X_par = np.stack([caracteristicas(preprocesar(leer_gris(r))) for r in firmas.ruta])
    np.save(CACHE_PAR, X_par)
pd.DataFrame(X_par, columns=NOMBRES_CAR).describe().T.round(3)

### 2.3 Patrón y puntuación de similitud
- **Patrón** del usuario: media $\mu_k$ y desviación típica $\sigma_k$ de cada característica $k$ en sus 10 firmas de referencia.
- Con solo 10 firmas $\sigma_k$ puede ser casi 0, así que se regulariza: $\sigma'_k = \sqrt{\sigma_k^2 + (\alpha\,s_k)^2}$, donde $s_k$ es la desviación global de la característica en desarrollo y $\alpha$ un hiperparámetro.
- **Puntuación** de una firma $x$: $\;s(x) = -\frac{1}{K}\sum_k \left|\frac{x_k-\mu_k}{\sigma'_k}\right|$ (cuanto mayor, más parecida).

In [ ]:
def puntuar_parametrico(X, df, usuarios_sel, s_global, alpha):
    filas = []
    for u in usuarios_sel:
        idx_u = df.index[df.usuario == u]
        ref = X[idx_u[df.loc[idx_u, "rol"] == "referencia"]]
        mu, sd = ref.mean(0), ref.std(0, ddof=1)
        sd = np.sqrt(sd**2 + (alpha * s_global)**2)
        for i in idx_u[df.loc[idx_u, "rol"] == "prueba"]:
            s = -np.mean(np.abs((X[i] - mu) / sd))
            filas.append(dict(usuario=u, firma=df.firma[i], tipo=df.tipo[i], puntuacion=s))
    return pd.DataFrame(filas)

# la desviación global se calcula SOLO con usuarios de desarrollo
mask_dev = firmas.usuario.isin(USR_DEV).values
S_GLOBAL = X_par[mask_dev].std(0)

### 2.4 Ajuste en desarrollo (hiperparámetro α y umbral global)

In [ ]:
res = []
for alpha in [0.0, 0.1, 0.25, 0.5, 1.0, 2.0, 4.0]:
    sc = puntuar_parametrico(X_par, firmas, USR_DEV, S_GLOBAL, alpha)
    e, t = eer(*separar(sc))
    res.append(dict(alpha=alpha, EER=100 * e, umbral=t))
res = pd.DataFrame(res); display(res)

ALPHA_PAR = res.loc[res.EER.idxmin(), "alpha"]
scores_par_dev = puntuar_parametrico(X_par, firmas, USR_DEV, S_GLOBAL, ALPHA_PAR)
EER_par_dev, UMBRAL_PAR = eer(*separar(scores_par_dev))  # umbral global fijado en desarrollo
print(f"α elegido = {ALPHA_PAR} | EER desarrollo = {100*EER_par_dev:.2f} % | UMBRAL FIJO = {UMBRAL_PAR:.4f}")

In [ ]:
g, f = separar(scores_par_dev)
fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
axs[0].hist(g, bins=40, alpha=0.6, density=True, label="Genuinas")
axs[0].hist(f, bins=40, alpha=0.6, density=True, label="Falsificaciones")
axs[0].axvline(UMBRAL_PAR, color="k", ls="--", label="Umbral")
axs[0].set_xlabel("Puntuación de similitud"); axs[0].legend(); axs[0].set_title("Distribución de puntuaciones (desarrollo)")
plot_det({"Paramétrico (dev)": scores_par_dev}, "DET paramétrico · desarrollo", ax=axs[1])
plt.tight_layout(); plt.show()

### 2.5 Resultado en evaluación (umbral y α fijos)

In [ ]:
scores_par_eval = puntuar_parametrico(X_par, firmas, USR_EVAL, S_GLOBAL, ALPHA_PAR)
resumen(scores_par_eval, UMBRAL_PAR, "Paramétrico · evaluación");

## 3. Sistema de aprendizaje automático (versión simplificada de SigNet)

**Artículo de referencia (Scopus):**
> Hafemann, L. G., Sabourin, R., & Oliveira, L. S. (2017). *Learning features for offline handwritten signature verification using deep convolutional neural networks.* **Pattern Recognition, 70**, 163–176. https://doi.org/10.1016/j.patcog.2017.05.012

**Idea del artículo.** En lugar de diseñar características a mano, se **aprenden** con una red convolucional (CNN). La red se entrena con usuarios de desarrollo para una tarea auxiliar, **reconocer quién ha firmado** (clasificación de escritores). Así aprende a representar los rasgos que diferencian la escritura de cada persona. Después se descarta la capa de clasificación y la red se usa como **extractor de características** para usuarios que nunca ha visto (enfoque *writer-independent*). Por último, cada firma se compara con las referencias del usuario.

**Simplificaciones respecto al artículo:**

| Artículo (SigNet) | Esta práctica |
|---|---|
| CNN propia entrenada desde cero con GPDS (~500 usuarios) | **ResNet18 preentrenada en ImageNet** (torchvision) con **ajuste fino** en 20 usuarios de CEDAR |
| Un SVM por usuario sobre las características | **Similitud coseno** con las 10 referencias (sin entrenar nada por usuario) |
| Variante SigNet-F con falsificaciones (parámetro λ) | Se probó y empeoraba con tan pocos usuarios, así que se usa solo la clasificación de escritores |

**Protocolo** (todo con usuarios de desarrollo, sin tocar los de evaluación):
- Usuarios **1–20**: entrenamiento de la CNN (sus 24 firmas genuinas, 20 clases).
- Usuarios **21–30**: validación. Se elige el hiperparámetro y se fija el umbral con usuarios que la red no ha visto, que es la misma situación que tendrán los usuarios nuevos.
- Usuarios **31–43**: evaluación final.

**Hiperparámetro estudiado:** número de **épocas de ajuste fino** ∈ {0, 10, 30, 60}. Con 0 épocas la red solo usa lo aprendido en ImageNet. También se compara la forma de agregar las 10 similitudes: media o máximo.

Recursos: PyTorch y torchvision (pesos ImageNet de ResNet18).

In [ ]:
import torch, torch.nn as nn, torchvision
from torchvision import transforms as T

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
H_IMG, W_IMG = 150, 220          # tamaño de entrada (el mismo que en SigNet)
USR_TRAIN_CNN = [u for u in USR_DEV if u <= 20]
USR_VAL_CNN = [u for u in USR_DEV if u > 20]
EPOCAS_CANDIDATAS = [0, 10, 30, 60]
RUTA_MODELO = "modelo_signet_resnet18.pt"
print("Dispositivo:", DEVICE)

def preprocesar_cnn(img):
    """Preprocesado al estilo SigNet: fondo a 0 (Otsu), tinta invertida, recorte,
    relleno para mantener la proporción y redimensionado a H_IMG×W_IMG."""
    g = cv2.medianBlur(img, 3)
    _, b = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    inv = 255 - g
    inv[b == 0] = 0
    ys, xs = np.nonzero(b)
    c = inv[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    h, w = c.shape
    r = W_IMG / H_IMG
    if w / h > r:
        pad = int(w / r) - h; c = np.pad(c, ((pad // 2, pad - pad // 2), (0, 0)))
    else:
        pad = int(h * r) - w; c = np.pad(c, ((0, 0), (pad // 2, pad - pad // 2)))
    return cv2.resize(c, (W_IMG, H_IMG), interpolation=cv2.INTER_AREA)

MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
def a_tensor(imgs):
    x = torch.tensor(np.stack(imgs), dtype=torch.float32, device=DEVICE).unsqueeze(1) / 255.
    return (x.repeat(1, 3, 1, 1) - MEAN) / STD

IMGS_CNN = np.stack([preprocesar_cnn(leer_gris(r)) for r in firmas.ruta])
fig, axs = plt.subplots(1, 4, figsize=(14, 2.5))
for ax, i in zip(axs, [0, 1, 30, 31]):
    ax.imshow(IMGS_CNN[i], cmap="gray"); ax.set_title(f"u{firmas.usuario[i]} {firmas.tipo[i]} {firmas.firma[i]}"); ax.axis("off")
plt.suptitle("Entrada de la CNN"); plt.show()

In [ ]:
def crear_modelo(n_clases):
    m = torchvision.models.resnet18(weights="IMAGENET1K_V1")
    m.fc = nn.Linear(512, n_clases)   # capa de clasificación de escritores
    return m.to(DEVICE)

@torch.no_grad()
def extraer(modelo, imgs, lote=64):
    """Vector de 512 características (penúltima capa), normalizado a norma 1."""
    fc, modelo.fc = modelo.fc, nn.Identity()
    modelo.eval()
    F = torch.cat([modelo(a_tensor(imgs[k:k + lote])) for k in range(0, len(imgs), lote)])
    modelo.fc = fc
    return nn.functional.normalize(F, dim=1).cpu().numpy()

def puntuar_ml(F, df, usuarios_sel, agregacion="max"):
    """Puntuación = similitud coseno (media o máxima) con las 10 referencias del usuario."""
    filas = []
    for u in usuarios_sel:
        idx_u = df.index[df.usuario == u]
        R = F[idx_u[df.loc[idx_u, "rol"] == "referencia"]]
        for i in idx_u[df.loc[idx_u, "rol"] == "prueba"]:
            s = R @ F[i]
            s = s.max() if agregacion == "max" else s.mean()
            filas.append(dict(usuario=u, firma=df.firma[i], tipo=df.tipo[i], puntuacion=float(s)))
    return pd.DataFrame(filas)

### 3.1 Entrenamiento y estudio del hiperparámetro (épocas de ajuste fino)
Se entrena una sola vez hasta 60 épocas y se evalúa en validación (usuarios 21–30) al llegar a 0, 10, 30 y 60. El mejor modelo se guarda en disco para usarlo sin reentrenar en la evaluación y en la prueba final.

In [ ]:
torch.manual_seed(0); np.random.seed(0)
idx_tr = firmas.index[firmas.usuario.isin(USR_TRAIN_CNN) & (firmas.tipo == "genuina")].values
clase = {u: i for i, u in enumerate(USR_TRAIN_CNN)}
y_tr = torch.tensor([clase[u] for u in firmas.usuario[idx_tr]], device=DEVICE)

modelo = crear_modelo(len(clase))
aumento = T.RandomAffine(degrees=5, translate=(0.05, 0.05), scale=(0.9, 1.1), shear=5)
opt = torch.optim.Adam(modelo.parameters(), lr=1e-4, weight_decay=1e-4)

resultados_hp, mejor = [], None
for ep in range(max(EPOCAS_CANDIDATAS) + 1):
    if ep in EPOCAS_CANDIDATAS:
        F = extraer(modelo, IMGS_CNN)
        for agg in ["media", "max"]:
            e, t = eer(*separar(puntuar_ml(F, firmas, USR_VAL_CNN, agg)))
            resultados_hp.append(dict(epocas=ep, agregacion=agg, EER_val=100 * e, umbral=t))
            if mejor is None or e < mejor["EER"]:
                mejor = dict(EER=e, epocas=ep, agregacion=agg,
                             estado={k: v.detach().clone() for k, v in modelo.state_dict().items()})
        print(f"época {ep:2d} · EER val (media/max): "
              f"{resultados_hp[-2]['EER_val']:.2f} % / {resultados_hp[-1]['EER_val']:.2f} %")
    if ep == max(EPOCAS_CANDIDATAS):
        break
    modelo.train()
    perm = np.random.permutation(len(idx_tr))
    for k in range(0, len(perm), 32):
        bi = perm[k:k + 32]
        loss = nn.functional.cross_entropy(modelo(aumento(a_tensor(IMGS_CNN[idx_tr[bi]]))), y_tr[bi])
        opt.zero_grad(); loss.backward(); opt.step()

resultados_hp = pd.DataFrame(resultados_hp)
torch.save(dict(estado=mejor["estado"], n_clases=len(clase), epocas=mejor["epocas"],
                agregacion=mejor["agregacion"]), RUTA_MODELO)
print(f"Mejor configuración: {mejor['epocas']} épocas, agregación '{mejor['agregacion']}' "
      f"(EER val = {100*mejor['EER']:.2f} %) → guardado en {RUTA_MODELO}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for agg, d in resultados_hp.groupby("agregacion"):
    ax.plot(d.epocas, d.EER_val, "o-", lw=2, label=f"agregación: {agg}")
ax.set_xlabel("Épocas de ajuste fino"); ax.set_ylabel("EER validación (%)")
ax.set_title("Influencia del hiperparámetro"); ax.set_xticks(EPOCAS_CANDIDATAS); ax.grid(alpha=0.3); ax.legend()
plt.show()
resultados_hp

### 3.2 Modelo final, umbral fijo y evaluación
Se carga el mejor modelo y se fija el **umbral global** en los usuarios de validación (21–30). Ni el modelo ni el umbral se vuelven a tocar.

In [ ]:
ck = torch.load(RUTA_MODELO, map_location=DEVICE)
modelo_ml = crear_modelo(ck["n_clases"]); modelo_ml.load_state_dict(ck["estado"])
AGG_ML = ck["agregacion"]
F_ml = extraer(modelo_ml, IMGS_CNN)

scores_ml_dev = puntuar_ml(F_ml, firmas, USR_VAL_CNN, AGG_ML)
EER_ml_dev, UMBRAL_ML = eer(*separar(scores_ml_dev))
print(f"ML · validación: EER = {100*EER_ml_dev:.2f} % | UMBRAL FIJO = {UMBRAL_ML:.4f}")

scores_ml_eval = puntuar_ml(F_ml, firmas, USR_EVAL, AGG_ML)
resumen(scores_ml_eval, UMBRAL_ML, "ML (SigNet simplificado) · evaluación");

## 4. Comparación biométrica (usuarios de evaluación 31–43)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
plot_det({"Paramétrico OpenCV": scores_par_eval, "ML · SigNet simplificado": scores_ml_eval},
         "Curvas DET · usuarios de evaluación", ax=ax)
for df_s, t, c in [(scores_par_eval, UMBRAL_PAR, "C0"), (scores_ml_eval, UMBRAL_ML, "C1")]:
    far, frr = far_frr(*separar(df_s), t)
    ax.plot(norm.ppf(max(far, 1e-3)), norm.ppf(max(frr, 1e-3)), "s", color=c, ms=10, mec="k")
ax.text(0.02, 0.02, "■ = punto de trabajo con el umbral fijado en desarrollo", transform=ax.transAxes, fontsize=9)
plt.savefig("det_comparacion.png", dpi=200, bbox_inches="tight"); plt.show()

tabla = []
for nombre, df_s, t, e_dev in [("Paramétrico OpenCV", scores_par_eval, UMBRAL_PAR, EER_par_dev),
                               ("ML · SigNet simplificado", scores_ml_eval, UMBRAL_ML, EER_ml_dev)]:
    g, f = separar(df_s)
    e, _ = eer(g, f); far, frr = far_frr(g, f, t)
    tabla.append({"Sistema": nombre, "EER desarrollo (%)": 100 * e_dev, "EER evaluación (%)": 100 * e,
                  "Umbral fijo": t, "FAR (%)": 100 * far, "FRR (%)": 100 * frr})
tabla = pd.DataFrame(tabla).round(2); tabla

**Discusión** *(completar con vuestros resultados)*:
- ¿Qué sistema tiene menor EER? ¿Se mantiene la diferencia en todo el rango de la curva DET?
- ¿El umbral fijado en desarrollo da en evaluación una FAR y una FRR parecidas, o se desequilibra? Si se desequilibra, el umbral no generaliza perfectamente a usuarios nuevos.
- **¿Qué error es más importante?** En aplicaciones de **alta seguridad** (cheques, contratos, banca) es más grave la **FAR**: aceptar una falsificación supone fraude. Conviene subir el umbral aunque aumenten los rechazos de genuinos. En aplicaciones de **comodidad** o con verificación manual posterior importa más la **FRR**, porque rechazar al usuario auténtico deteriora el servicio.

## 5. Prueba final con 10 usuarios nuevos
Se usa el **mejor sistema según desarrollo**, con sus hiperparámetros, su modelo y su umbral **fijos**. Para cada usuario nuevo solo se construye el patrón con sus genuinas 1–10. Se genera el CSV con `usuario, firma, tipo, puntuacion` y se calculan los errores.

Uso el día de la prueba: descomprimir las firmas nuevas y cambiar las dos rutas de la celda siguiente.

In [ ]:
MEJOR_SISTEMA = "ML" if EER_ml_dev < EER_par_dev else "Parametrico"
print("Sistema elegido (según desarrollo):", MEJOR_SISTEMA)

def evaluar_usuarios_nuevos(dir_gen, dir_forg, salida_csv="puntuaciones_prueba_final.csv", sistema=MEJOR_SISTEMA, usuarios=None):
    df_new = listar_firmas(dir_gen, dir_forg)
    if usuarios is not None:
        df_new = df_new[df_new.usuario.isin(usuarios)].reset_index(drop=True)
    usrs = sorted(df_new.usuario.unique())
    if sistema == "ML":
        imgs = np.stack([preprocesar_cnn(leer_gris(r)) for r in df_new.ruta])
        sc = puntuar_ml(extraer(modelo_ml, imgs), df_new, usrs, AGG_ML); umbral = UMBRAL_ML
    else:
        X = np.stack([caracteristicas(preprocesar(leer_gris(r))) for r in df_new.ruta])
        sc = puntuar_parametrico(X, df_new, usrs, S_GLOBAL, ALPHA_PAR); umbral = UMBRAL_PAR
    sc["decision"] = np.where(sc.puntuacion >= umbral, "aceptada", "rechazada")
    sc.to_csv(salida_csv, index=False)
    print(f"{len(usrs)} usuarios, {len(sc)} firmas puntuadas → {salida_csv}")
    resumen(sc, umbral, f"Prueba final ({sistema})")
    return sc

# Ensayo general con los usuarios de evaluación.
# EL DÍA DE LA PRUEBA: poner las rutas de las firmas nuevas y quitar usuarios=USR_EVAL
DIR_GEN_NUEVOS, DIR_FORG_NUEVOS = DIR_GEN, DIR_FORG
sc_final = evaluar_usuarios_nuevos(DIR_GEN_NUEVOS, DIR_FORG_NUEVOS, "puntuaciones_ensayo.csv", usuarios=USR_EVAL)
sc_final.head()

## Referencias y declaración de uso de IA
- Hafemann, L. G., Sabourin, R., & Oliveira, L. S. (2017). Learning features for offline handwritten signature verification using deep convolutional neural networks. *Pattern Recognition, 70*, 163–176. https://doi.org/10.1016/j.patcog.2017.05.012
- He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep residual learning for image recognition. *CVPR*, 770–778. (arquitectura ResNet18)
- Base de datos CEDAR de firmas manuscritas (Center of Excellence for Document Analysis and Recognition, Univ. at Buffalo).
- Bibliotecas: OpenCV, NumPy, pandas, SciPy, Matplotlib, PyTorch y torchvision.

**Declaración de uso de IA:** se ha utilizado un asistente de IA (Claude, Anthropic) como apoyo para estructurar el notebook y redactar parte del código y de las explicaciones. Los autores han revisado, ejecutado y comprendido todo el código y son responsables de los resultados y conclusiones. *(Ajustar a vuestro uso real.)*